In [3]:
import gymnasium as gym
import numpy as np

## Value iterator

In [4]:
def value_iteration(entorno, gamma=0.99, theta=1e-8):
    """
    Implementa el algoritmo Value Iteration (Iteración de Valor).

    Fundamento teórico:
    -------------------
    Aplica la ecuación de Bellman iterativamente sobre todos los estados
    hasta convergencia. En cada barrida actualiza:

        V(s) = max_a { Σ P(s'|s,a) · [R + γ · V(s')] }

    Al converger, extrae la política óptima:
        π*(s) = argmax_a Q(s,a)

    Parámetros:
    -----------
    entorno : gym.Env  — entorno Gym con acceso a entorno.P (modelo del entorno)
    gamma   : float    — factor de descuento (0 < γ ≤ 1)
    theta   : float    — criterio de convergencia (umbral de cambio mínimo)

    Retorna:
    --------
    V       : np.ndarray — función de valor óptima para cada estado
    politica: np.ndarray — política óptima (acción por estado)
    iteraciones: int     — cantidad de iteraciones hasta convergencia
    """

    # ── Paso 1: Obtener dimensiones del espacio de estados y acciones ──────
    num_estados = entorno.observation_space.n
    num_acciones = entorno.action_space.n

    # ── Paso 2: Inicializar la función de valor en 0 para todos los estados ─
    V = np.zeros(num_estados)  # V[s] = valor del estado s

    # Acceder a la dinámica del entorno (env.unwrapped.P en Gymnasium moderno)
    # P[estado][accion] = lista de (probabilidad, sig_estado, recompensa, terminal)
    modelo = entorno.unwrapped.P

    iteraciones = 0  # Contador de barridas (sweeps)

    # ── Paso 3: Bucle principal hasta convergencia ─────────────────────────
    while True:
        delta = 0.0  # Máximo cambio en esta barrida (criterio de parada)
        iteraciones += 1

        # Recorrer todos los estados
        for estado in range(num_estados):
            valor_anterior = V[estado]  # Guardar valor previo para calcular Δ

            # Calcular el valor Q para cada acción posible en este estado
            # modelo[estado][accion] = lista de (prob, sig_estado, recompensa, terminal)
            valores_q = np.zeros(num_acciones)
            for accion in range(num_acciones):
                for prob_transicion, sig_estado, recompensa, es_terminal in modelo[
                    estado
                ][accion]:
                    # Ecuación de Bellman: suma sobre estados siguientes ponderada por probabilidad
                    # En modo determinístico: prob_transicion = 1.0 siempre
                    # En modo estocástico:   prob_transicion < 1.0 (el entorno puede "resbalar")
                    valores_q[accion] += prob_transicion * (
                        recompensa + gamma * V[sig_estado]
                    )

            # Actualizar V(s) con el máximo Q-valor (acción óptima)
            V[estado] = np.max(valores_q)

            # Actualizar el cambio máximo de esta barrida
            delta = max(delta, abs(valor_anterior - V[estado]))

        # ── Verificar convergencia: si Δ < θ, detener ─────────────────────
        if delta < theta:
            break

    # ── Paso 4: Extraer la política óptima a partir de V ──────────────────
    politica = np.zeros(num_estados, dtype=int)
    for estado in range(num_estados):
        # Q(s,a) = Σ P(s'|s,a) · [R + γ · V(s')]
        valores_q = np.zeros(num_acciones)
        for accion in range(num_acciones):
            for prob_transicion, sig_estado, recompensa, es_terminal in modelo[estado][
                accion
            ]:
                valores_q[accion] += prob_transicion * (
                    recompensa + gamma * V[sig_estado]
                )

        # La política elige la acción que maximiza Q(s,a)
        politica[estado] = np.argmax(valores_q)

# Q-LEARNING

In [ ]:
def aprendizaje_q(
    entorno,
    episodios=10000,
    alfa=0.1,
    gamma=0.99,
    epsilon=1.0,
    decaimiento_epsilon=0.999,):

    
    num_estados = entorno.observation_space.n
    num_acciones = entorno.action_space.n

    Q = np.zeros((num_estados, num_acciones))
    recompensas_por_episodio = []

    for episodio in range(episodios):

        info_estado = entorno.reset()
        estado = info_estado[0]

        recompensa_total = 0
        terminado = False

        while not terminado:

            if np.random.uniform(0, 1) < epsilon:
                accion = entorno.action_space.sample() 
            else:
                accion = np.argmax(Q[estado, :])  

            resultado_accion = entorno.step(accion)

            siguiente_estado, recompensa, finalizado, truncado, _ = resultado_accion
            terminado = finalizado or truncado

            # 3. Actualizar Q mediante la ecuación de aprendizaje (Diferencia Temporal)
            # Calculamos el objetivo DT (Diferencia Temporal)
            # mejor_siguiente_accion = np.argmax(Q[siguiente_estado, :])
            # objetivo_dt = recompensa + gamma * Q[siguiente_estado, mejor_siguiente_accion] * (not terminado)
            # error_dt = objetivo_dt - Q[estado, accion]
            # Q[estado, accion] += alfa * error_dt
            # ESTA ERA LA FORMA ANTERIOR DE CALCULAR, ES LO MISMO LA DE ABAJO PERO EN UNA LINEA

            Q[estado, accion] += alfa * (recompensa + gamma * np.max(Q[siguiente_estado, :]) * (not terminado) - Q[estado, accion])

            estado = siguiente_estado
            recompensa_total += recompensa

        
        epsilon = max(0.01, epsilon * decaimiento_epsilon)
        recompensas_por_episodio.append(recompensa_total)


    politica = np.argmax(Q, axis=1)

    return Q, politica, recompensas_por_episodio


# DOS COSAS A TENER EN CUENTA
# la ecuacion q puede estar implementada en una linea o en varias, decidir eso
# el decaimiento de epsilon se puede hacer al comienzo como en el pseudocodigo pero conlleva que
# o se evite en el primer episodio o que epsilon no sea 100 en el primer intento, lo cual no se si es correcto

# CART POLE

Tiene dos variantes v0 y v1

el agente tiene que mantenerse con el palo en equilibrio lo mas que pueda
 - en v0 -> 200 pasos y umbral de exito 200
 - en v1 -> 500 pasos y umbal de exito 500

dos formas de recompensa
 1. El agente gana puntos por cada paso que sobrevive
 2. El agente no gana puntos pero pierde si se muere

dos formas de que en teoria tienen el mismo resultado pero estaria interesante comparar ambos

In [ ]:

env = gym.make("CartPole-v1", render_mode="human")
env
# <TimeLimit<OrderEnforcing<PassiveEnvChecker<CartPoleEnv<CartPole-v1>>>>>
env.reset(seed=123, options={"low": -0.1, "high": 0.1})  # default low=-0.05, high=0.05
# (array([ 0.03647037, -0.0892358 , -0.05592803, -0.06312564], dtype=float32), {})

observation, info = env.reset()
# observation: what the agent can "see" 
# info: extra debugging information (usually not needed for basic learning)

print(f"Starting observation: {observation}")

Starting observation: [-0.03240941  0.03120945  0.0423345  -0.02234256]


: 